# Βήμα 9 – Σύστημα Α: βελτίωση του κανόνα απόφασης
Το notebook 11 έδειξε δύο αδυναμίες:
- η κατηγορία **Analysis** προβλέπεται πολύ συχνά και σχεδόν πάντα λάθος (precision 6,8%)·
- το **35% των επιθέσεων υψηλής σοβαρότητας** χαρακτηρίζεται Low.

Εδώ **δεν αλλάζουμε το δίκτυο** (καμία εκπαίδευση). Αλλάζουμε μόνο το πώς μετατρέπουμε τις πιθανότητές του σε τελική απόφαση.

1. Πόσο μπορεί να βελτιωθεί; (ανώτατο όριο από τα ίδια τα δεδομένα)
2. Τύπος επίθεσης: διόρθωση της προτίμησης προς τις σπάνιες κλάσεις.
3. Σοβαρότητα: από την αναμενόμενη σοβαρότητα (την ίδια ποσότητα με το risk score) αντί από τον προβλεπόμενο τύπο.
4. Πριν / μετά στο test.

Όλες οι ρυθμίσεις γίνονται **στο validation**. Χρόνος: 1–3 λεπτά.

In [ ]:
import json
import numpy as np
import pandas as pd
import joblib
import matplotlib.pyplot as plt
import seaborn as sns
import keras
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix

DATA, MODELS, FIG = "../data/", "../models/", "../figures/"
THRESHOLD = 0.5            # κατώφλι επίθεσης (notebook 11)
UNDER_COST = 2.0           # ΠΑΡΑΔΟΧΗ: η υποεκτίμηση της σοβαρότητας «κοστίζει» διπλά από την υπερεκτίμηση (ανά επίπεδο)

with open(MODELS + "unsw_meta.json", encoding="utf-8") as f:
    meta = json.load(f)
with open(MODELS + "unsw_8num_tuned_config.json", encoding="utf-8") as f:
    config = json.load(f)
ATTACK_CLASSES, SEVERITY_CLASSES = meta["attack_classes"], meta["severity_classes"]
SEV_WEIGHTS = np.array([meta["severity_weight"][c] for c in SEVERITY_CLASSES])
SEV_OF_ATTACK = np.array([SEVERITY_CLASSES.index(meta["severity_map"][c]) for c in ATTACK_CLASSES])
FEATS_8 = config["features"]

train_full = pd.read_csv(DATA + "UNSW_NB15_train-set.csv")
test_df = pd.read_csv(DATA + "UNSW_NB15_test-set.csv")
for df in (train_full, test_df):
    df["attack_cat"] = df["attack_cat"].str.strip()
    df["severity"] = df["attack_cat"].map(meta["severity_map"])
if meta["drop_duplicates"]:
    train_full = train_full.drop_duplicates(subset=[c for c in train_full.columns if c != "id"])
train_df, val_df = train_test_split(train_full, test_size=0.15, stratify=train_full["attack_cat"], random_state=meta["random_state"])
frames = {"train": train_df, "val": val_df, "test": test_df}

def encode_targets(df):
    return {"label": df["label"].to_numpy("int64"),
            "attack_cat": df["attack_cat"].map({c: i for i, c in enumerate(ATTACK_CLASSES)}).to_numpy("int64"),
            "severity": df["severity"].map({c: i for i, c in enumerate(SEVERITY_CLASSES)}).to_numpy("int64")}
y = {s: encode_targets(df) for s, df in frames.items()}

preprocessor = joblib.load(MODELS + "unsw_preprocessor_8num_tuned.joblib")
model = keras.models.load_model(MODELS + "unsw_multitask_8num_tuned.keras", compile=False)
prob = {}
for s in ("val", "test"):
    p_label, p_attack, p_sev = model.predict(preprocessor.transform(frames[s]).astype("float32"), batch_size=1024, verbose=0)
    prob[s] = {"label": p_label.ravel(), "attack": p_attack, "sev": p_sev}
print("OK:", {s: prob[s]["attack"].shape for s in prob})

## 9.1 Πόσο μπορεί να βελτιωθεί; Ανώτατο όριο από τα δεδομένα
Αν δύο ροές έχουν **ακριβώς ίδιες τιμές** στα 8 χαρακτηριστικά αλλά διαφορετικό τύπο, κανένα μοντέλο δεν μπορεί να τις ξεχωρίσει: θα δώσει και στις δύο την ίδια απάντηση.

Για κάθε ομάδα ίδιων ροών στο test, το καλύτερο που μπορεί να κάνει **οποιοσδήποτε** ταξινομητής είναι να διαλέξει τον συχνότερο τύπο της ομάδας.
Αυτό δίνει ένα (αισιόδοξο) ανώτατο όριο· ό,τι λείπει από εκεί και πάνω δεν διορθώνεται με αυτά τα χαρακτηριστικά.

In [ ]:
def ceiling(df, target, classes):
    c = df.groupby(FEATS_8 + [target], sort=False).size().rename("n").reset_index()
    grp = c.groupby(FEATS_8, sort=False)["n"]
    c["types_in_group"] = grp.transform("size")
    c["is_major"] = False
    c.loc[grp.idxmax(), "is_major"] = True
    c["ambiguous"] = c["n"] * (c["types_in_group"] > 1)
    c["major"] = c["n"] * c["is_major"]
    by = c.groupby(target)[["n", "ambiguous", "major"]].sum().reindex(classes)
    out = pd.DataFrame({"πλήθος": by["n"].astype(int),
                        "ίδια χαρακτ. με άλλη κλάση": by["ambiguous"] / by["n"],
                        "ανώτατο όριο recall": by["major"] / by["n"]})
    return out, by["major"].sum() / by["n"].sum()

def decide(p_label, p_attack, threshold=THRESHOLD):
    is_attack = p_label >= threshold
    attack = np.where(is_attack, p_attack[:, 1:].argmax(1) + 1, 0)
    return is_attack.astype(int), attack, SEV_OF_ATTACK[attack]

yt = y["test"]
lab0, att0, sev0 = decide(prob["test"]["label"], prob["test"]["attack"])

ceil_type, ceil_type_acc = ceiling(test_df, "attack_cat", ATTACK_CLASSES)
ceil_type["recall τώρα"] = recall_score(yt["attack_cat"], att0, labels=range(len(ATTACK_CLASSES)), average=None, zero_division=0)
print("ΤΥΠΟΣ ΕΠΙΘΕΣΗΣ (test)"); print(ceil_type.round(3).to_string())
print(f"Ανώτατο όριο ακρίβειας: {ceil_type_acc:.4f} | τώρα: {accuracy_score(yt['attack_cat'], att0):.4f}\n")

ceil_sev, ceil_sev_acc = ceiling(test_df, "severity", SEVERITY_CLASSES)
ceil_sev["recall τώρα"] = recall_score(yt["severity"], sev0, labels=range(len(SEVERITY_CLASSES)), average=None, zero_division=0)
print("ΣΟΒΑΡΟΤΗΤΑ (test)"); print(ceil_sev.round(3).to_string())
print(f"Ανώτατο όριο ακρίβειας: {ceil_sev_acc:.4f} | τώρα: {accuracy_score(yt['severity'], sev0):.4f}")

## 9.2 Τύπος επίθεσης: διόρθωση της προτίμησης προς τις σπάνιες κλάσεις
Στην εκπαίδευση δώσαμε μεγαλύτερο βάρος στις σπάνιες κλάσεις (p = 0.25), για να μην τις αγνοεί το δίκτυο. Το τίμημα: όταν διστάζει, «γέρνει» προς αυτές.

Στην απόφαση διαιρούμε την πιθανότητα κάθε τύπου με `βάρος^τ`:
- `τ = 0` → η απόφαση όπως είναι τώρα·
- `τ = 1` → αναιρείται περίπου όλη η ώθηση των βαρών·
- ενδιάμεσες ή μεγαλύτερες τιμές → κάτι ανάμεσα ή ακόμα πιο «συντηρητικό».

Το `τ` διαλέγεται στο validation με το ίδιο κριτήριο που χρησιμοποιούμε παντού (macro-F1 του τύπου επίθεσης).

In [ ]:
def class_weights(y_cls, n_classes, power):
    counts = np.bincount(y_cls, minlength=n_classes).astype("float64")
    w = (len(y_cls) / (n_classes * np.maximum(counts, 1))) ** power
    return w / np.average(w, weights=counts)

W_ATTACK = class_weights(y["train"]["attack_cat"], len(ATTACK_CLASSES), config["class_weight_power"])
print("Βάρη εκπαίδευσης:", {c: round(float(v), 2) for c, v in zip(ATTACK_CLASSES, W_ATTACK)})

def decide_type(split, tau):
    p = prob[split]
    is_attack = p["label"] >= THRESHOLD
    scores = p["attack"][:, 1:] / W_ATTACK[1:] ** tau
    return np.where(is_attack, scores.argmax(1) + 1, 0)

A = ATTACK_CLASSES.index("Analysis")
def type_metrics(split, att):
    t = y[split]["attack_cat"]
    labels = range(len(ATTACK_CLASSES))
    prec = precision_score(t, att, labels=labels, average=None, zero_division=0)
    rec = recall_score(t, att, labels=labels, average=None, zero_division=0)
    sev = SEV_OF_ATTACK[att]
    high = y[split]["severity"] == 3
    return {"attack_cat acc": accuracy_score(t, att), "attack_cat macro-F1": f1_score(t, att, average="macro"),
            "Analysis precision": prec[A], "Analysis recall": rec[A],
            "DoS recall": rec[ATTACK_CLASSES.index("DoS")], "Backdoor recall": rec[ATTACK_CLASSES.index("Backdoor")],
            "προβλέψεις Analysis": int((att == A).sum()),
            "High → Low (από τον τύπο)": ((sev == 1) & high).sum() / high.sum()}

TAUS = np.round(np.arange(0, 3.01, 0.25), 2)
tau_val = pd.DataFrame([type_metrics("val", decide_type("val", t)) for t in TAUS], index=TAUS)
tau_val.index.name = "τ"
TAU = float(tau_val["attack_cat macro-F1"].idxmax())
print("\nVALIDATION"); print(tau_val.round(4).to_string())
print(f"\nΕπιλογή στο validation (μέγιστο macro-F1): τ = {TAU}")

fig, axes = plt.subplots(1, 2, figsize=(11.5, 4.2))
axes[0].plot(TAUS, tau_val["attack_cat acc"], color="#2a78d6", lw=2, marker="o", ms=4, label="attack_cat acc")
axes[0].plot(TAUS, tau_val["attack_cat macro-F1"], color="#eb6834", lw=2, marker="o", ms=4, label="attack_cat macro-F1")
axes[1].plot(TAUS, tau_val["Analysis precision"], color="#2a78d6", lw=2, marker="o", ms=4, label="Analysis precision")
axes[1].plot(TAUS, tau_val["Analysis recall"], color="#eb6834", lw=2, marker="o", ms=4, label="Analysis recall")
axes[1].plot(TAUS, tau_val["High → Low (από τον τύπο)"], color="#1baf7a", lw=2, marker="o", ms=4, label="High που βγαίνουν Low")
for ax, title in zip(axes, ["Τύπος επίθεσης συνολικά (validation)", "Η κλάση Analysis και η υποεκτίμηση (validation)"]):
    ax.axvline(TAU, color="gray", ls=":", lw=1.4, label=f"επιλογή: τ = {TAU}")
    ax.set_xlabel("τ (0 = όπως τώρα)"); ax.set_title(title); ax.set_ylim(0, 1.02)
    ax.grid(alpha=0.3); ax.spines[["top", "right"]].set_visible(False)
    ax.legend(fontsize=8, ncol=2, loc="upper center", bbox_to_anchor=(0.5, -0.17), frameon=False)
plt.tight_layout(); plt.savefig(FIG + "33_unsw_prior_correction.png", dpi=200, bbox_inches="tight"); plt.show()

In [ ]:
att_new = decide_type("test", TAU)
type_cmp = pd.DataFrame({"τώρα (τ = 0)": type_metrics("test", att0), f"με διόρθωση (τ = {TAU})": type_metrics("test", att_new)})
print("TEST"); print(type_cmp.round(4).to_string())

labels = range(len(ATTACK_CLASSES))
per_type = pd.DataFrame({
    "πλήθος": np.bincount(yt["attack_cat"], minlength=len(ATTACK_CLASSES)),
    "precision τώρα": precision_score(yt["attack_cat"], att0, labels=labels, average=None, zero_division=0),
    "precision μετά": precision_score(yt["attack_cat"], att_new, labels=labels, average=None, zero_division=0),
    "recall τώρα": recall_score(yt["attack_cat"], att0, labels=labels, average=None, zero_division=0),
    "recall μετά": recall_score(yt["attack_cat"], att_new, labels=labels, average=None, zero_division=0),
    "ανώτατο όριο recall": ceil_type["ανώτατο όριο recall"].to_numpy()}, index=ATTACK_CLASSES)
print("\nΑνά τύπο (test):"); print(per_type.round(3).to_string())
per_type.to_csv(MODELS + "unsw_decision_type_before_after.csv")

## 9.3 Σοβαρότητα από την αναμενόμενη σοβαρότητα
Τώρα η σοβαρότητα βγαίνει από τον **έναν** πιο πιθανό τύπο: αν αυτός είναι η Analysis, η απάντηση είναι Low, ακόμα κι αν το δίκτυο διστάζει πολύ.

Το δίκτυο όμως έχει και ξεχωριστή έξοδο σοβαρότητας, που δίνει πιθανότητες. Αν π.χ. δίνει «40% Low, 25% Medium, 35% High», το 60% της πιθανότητας είναι σε κάτι σοβαρότερο από Low.
Χρησιμοποιούμε λοιπόν την **αναμενόμενη σοβαρότητα** `E = Σ P(σοβαρότητα) · βάρος` (Low 0.33, Medium 0.66, High 1.0· στο παράδειγμα E = 0.65) – είναι η ίδια ποσότητα που μπαίνει στο risk score – και τη χωρίζουμε σε τρεις ζώνες με δύο όρια:

| Κανόνας | Όρια | Τι υποθέτει |
|---|---|---|
| Α – τρέχων | – | σοβαρότητα = αυτή του πιο πιθανού τύπου |
| Α′ – τρέχων με τη διόρθωση τ | – | όπως ο Α, με τον τύπο της ενότητας 9.2 |
| Β – αναμενόμενη, φυσικά όρια | 0.495 και 0.83 (τα μέσα μεταξύ των βαρών) | τίποτα· δεν ρυθμίζεται |
| Γ – αναμενόμενη, ρυθμισμένα όρια | διαλέγονται στο validation | ότι η υποεκτίμηση κοστίζει διπλά από την υπερεκτίμηση (`UNDER_COST`) |

Με τους Β και Γ η σοβαρότητα **δεν προκύπτει πια από τον τύπο**: ο τύπος είναι η καλύτερη εκτίμηση, η σοβαρότητα λαμβάνει υπόψη και την αβεβαιότητα.

In [ ]:
def expected_severity(split):
    p = prob[split]["sev"][:, 1:]
    return (p * SEV_WEIGHTS[1:]).sum(1) / np.clip(p.sum(1), 1e-9, None)

def sev_from_expected(split, c1, c2):
    e = expected_severity(split)
    level = np.where(e < c1, 1, np.where(e < c2, 2, 3))
    return np.where(prob[split]["label"] >= THRESHOLD, level, 0)

def sev_metrics(split, pred):
    t = y[split]["severity"]
    diff = pred - t
    return {"severity acc": accuracy_score(t, pred), "severity macro-F1": f1_score(t, pred, average="macro"),
            "High → Low": ((pred == 1) & (t == 3)).sum() / (t == 3).sum(),
            "Medium → Low": ((pred == 1) & (t == 2)).sum() / (t == 2).sum(),
            "Low → Medium/High": ((pred >= 2) & (t == 1)).sum() / (t == 1).sum(),
            "υποεκτίμηση (όλες οι ροές)": (diff < 0).mean(), "υπερεκτίμηση (όλες οι ροές)": (diff > 0).mean(),
            "μέσο κόστος": np.where(diff < 0, -diff * UNDER_COST, diff).mean()}

NATURAL = (round(float(SEV_WEIGHTS[1] + SEV_WEIGHTS[2]) / 2, 3), round(float(SEV_WEIGHTS[2] + SEV_WEIGHTS[3]) / 2, 3))
best, best_cost = NATURAL, np.inf
for c1 in np.round(np.arange(0.35, 0.661, 0.01), 2):
    for c2 in np.round(np.arange(0.67, 0.991, 0.01), 2):
        cost = sev_metrics("val", sev_from_expected("val", c1, c2))["μέσο κόστος"]
        if cost < best_cost - 1e-12:
            best, best_cost = (float(c1), float(c2)), cost
TUNED = best
print(f"Φυσικά όρια (κανόνας Β): {NATURAL} | ρυθμισμένα στο validation (κανόνας Γ): {TUNED}")

def all_rules(split):
    lab, att, sev = decide(prob[split]["label"], prob[split]["attack"])
    return {"Α – τρέχων": sev,
            f"Α′ – τρέχων, τ = {TAU}": SEV_OF_ATTACK[decide_type(split, TAU)],
            "Β – αναμενόμενη, φυσικά όρια": sev_from_expected(split, *NATURAL),
            "Γ – αναμενόμενη, ρυθμισμένα όρια": sev_from_expected(split, *TUNED)}

sev_val = pd.DataFrame({k: sev_metrics("val", v) for k, v in all_rules("val").items()})
rules_test = all_rules("test")
sev_test = pd.DataFrame({k: sev_metrics("test", v) for k, v in rules_test.items()})
print("\nVALIDATION"); print(sev_val.round(4).to_string())
print("\nTEST"); print(sev_test.round(4).to_string())
pd.concat({"validation": sev_val, "test": sev_test}, axis=1).round(4).to_csv(MODELS + "unsw_decision_severity_rules.csv")

## 9.4 Πίνακες σύγχυσης της σοβαρότητας: πριν και μετά (test)
Κάθε γραμμή είναι μια πραγματική σοβαρότητα. Το κελί που μας ενδιαφέρει περισσότερο είναι το **High → Low** (κάτω αριστερά στη στήλη Low).

In [ ]:
show = ["Α – τρέχων", "Β – αναμενόμενη, φυσικά όρια", "Γ – αναμενόμενη, ρυθμισμένα όρια"]
fig, axes = plt.subplots(1, 3, figsize=(15.5, 4.8))
for ax, name in zip(axes, show):
    cm = confusion_matrix(yt["severity"], rules_test[name], labels=range(len(SEVERITY_CLASSES)))
    cmn = cm / cm.sum(1, keepdims=True).clip(min=1)
    annot = np.array([[f"{cmn[i, j]:.0%}\n{cm[i, j]}" for j in range(4)] for i in range(4)])
    sns.heatmap(cmn, annot=annot, fmt="", cmap="Blues", vmin=0, vmax=1, cbar=False,
                xticklabels=SEVERITY_CLASSES, yticklabels=SEVERITY_CLASSES, annot_kws={"size": 9}, ax=ax)
    ax.set_title(name, fontsize=11); ax.set_xlabel("Πρόβλεψη"); ax.set_ylabel("Πραγματικό")
    plt.setp(ax.get_yticklabels(), rotation=0)
fig.suptitle("Σοβαρότητα – τρεις κανόνες απόφασης στο ίδιο δίκτυο (test)", y=1.02)
plt.tight_layout(); plt.savefig(FIG + "34_unsw_severity_rules.png", dpi=200, bbox_inches="tight"); plt.show()

## 9.5 Αποθήκευση των ρυθμίσεων
Αποθηκεύονται όλες οι υποψήφιες ρυθμίσεις· ποια θα χρησιμοποιήσει η συνάρτηση πρόβλεψης το αποφασίζουμε αφού δούμε τα αποτελέσματα.

In [ ]:
rule = {"threshold": THRESHOLD, "tau_prior_correction": TAU, "attack_class_weights": [float(w) for w in W_ATTACK],
        "severity_cuts_natural": list(NATURAL), "severity_cuts_tuned": list(TUNED), "under_cost": UNDER_COST,
        "note": "Όλες οι τιμές επιλέχθηκαν στο validation. Η επιλογή κανόνα δεν έχει γίνει ακόμα."}
with open(MODELS + "unsw_decision_rule.json", "w", encoding="utf-8") as f:
    json.dump(rule, f, ensure_ascii=False, indent=2)
print(json.dumps(rule, ensure_ascii=False, indent=2))

## Τι κοιτάμε στα αποτελέσματα
- **9.1:** αν το «ανώτατο όριο recall» των DoS και Backdoor είναι χαμηλό, ο τύπος τους δεν διορθώνεται με αυτά τα χαρακτηριστικά, ό,τι κι αν κάνουμε.
- **9.2:** αν το `τ` που επιλέχθηκε είναι 0, η διόρθωση δεν βοηθά με το κριτήριό μας και μένουμε όπως είμαστε.
- **9.3:** οι κανόνες Β και Γ πρέπει να μειώνουν το «High → Low»· το ερώτημα είναι πόσο ανεβαίνει η υπερεκτίμηση σε αντάλλαγμα.